In [8]:
using SymPy
using PyCall
using BenchmarkTools
cd("c:/FD")
pushfirst!(PyCall.PyVector(PyCall.pyimport("sys")."path"), pwd())
importlib = pyimport("importlib")
fH_func = pyimport("fH_func")
importlib.reload(fH_func)
fH_class = pyimport("fH_class")
importlib.reload(fH_class)
grid_data = pyimport("grid_data")
importlib.reload(grid_data)
func_terms = fH_class.func_terms_class(n_max=5)
#func_terms = fH_func

kx = convert(Float64, grid_data."kx_val")
ky = convert(Float64, grid_data."ky_val")
kz = convert(Float64, grid_data."kz_val")
b  = convert(Float64, grid_data."b_val")
h  = convert(Float64, grid_data."h_val")
mu = convert(Float64, grid_data."mu_val")
x_full = convert(Array{Float64}, grid_data."x")
y_full = convert(Array{Float64}, grid_data."y")
z_full = convert(Array{Float64}, grid_data."z")

sympy = pyimport("sympy")

using MacroTools

function devectorize(expr)
    MacroTools.postwalk(expr) do x
        if @capture(x, a_.b_)
            return :($a.$b)
        elseif x isa Expr && x.head == :.
            return Expr(:call, x.args[1], x.args[2:end]...)
        elseif x isa Expr && x.head == :call
            if x.args[1] isa Symbol
                op_str = string(x.args[1])
                if startswith(op_str, ".")
                    new_op = Symbol(op_str[2:end])
                    return Expr(:call, new_op, x.args[2:end]...)
                end
            end
        end
        return x
    end
end

# ========== 预计算 exp ==========
function precompute_exp_terms(x, y, z, terms_list)
    n_points = length(x)
    n_terms = length(terms_list)
    exp_values = zeros(Float64, n_points, n_terms)
    
    for (term_idx, term) in enumerate(terms_list)
        exp_part, _ = term
        exp_julia_code = sympy.julia_code(exp_part)
        exp_func = eval(Meta.parse("(x, y, z) -> " * exp_julia_code))
        
        for i in 1:n_points
            exp_values[i, term_idx] = Base.invokelatest(exp_func, x[i], y[i], z[i])
        end
    end
    
    return exp_values
end

# ========== 生成完整的函数代码并保存到文件 ==========
poly_exprs_cos = [sympy.julia_code(poly_part) for (_, poly_part) in func_terms.terms1]
poly_exprs_sin = [sympy.julia_code(poly_part) for (_, poly_part) in func_terms.terms2]

# 生成多项式函数代码
function generate_poly_functions(poly_exprs, prefix)
    code = ""
    for (i, expr) in enumerate(poly_exprs)
        parsed = Meta.parse(expr)
        devec = devectorize(parsed)
        func_code = """
        @inline function $(prefix)_$i(x::Float64, y::Float64, z::Float64,
                                       kx::Float64, ky::Float64, kz::Float64,
                                       b::Float64, h::Float64, mu::Float64)
            return $(devec)
        end
        
        """
        code *= func_code
    end
    return code
end

# 生成完整的模块代码
full_code = """
module OptimizedPolynomials

# 生成所有 cos 多项式
$(generate_poly_functions(poly_exprs_cos, "poly_cos"))

# 生成所有 sin 多项式
$(generate_poly_functions(poly_exprs_sin, "poly_sin"))

# 主计算函数
function compute_optimized!(result::Vector{Float64},
                           exp_cos::Matrix{Float64}, exp_sin::Matrix{Float64},
                           cos_phase::Vector{Float64}, sin_phase::Vector{Float64},
                           x::Vector{Float64}, y::Vector{Float64}, z::Vector{Float64}, 
                           kx::Float64, ky::Float64, kz::Float64, 
                           b::Float64, h::Float64, mu::Float64)
    n_points = length(x)
    n_cos = size(exp_cos, 2)
    n_sin = size(exp_sin, 2)
    
    @inbounds for i in 1:n_points
        xi, yi, zi = x[i], y[i], z[i]
        sum_cos = 0.0
        sum_sin = 0.0
        
        # 计算 cos 项
        $(join(["sum_cos += exp_cos[i, $k] * poly_cos_$k(xi, yi, zi, kx, ky, kz, b, h, mu)" 
                for k in 1:length(poly_exprs_cos)], "\n        "))
        
        # 计算 sin 项
        $(join(["sum_sin += exp_sin[i, $k] * poly_sin_$k(xi, yi, zi, kx, ky, kz, b, h, mu)" 
                for k in 1:length(poly_exprs_sin)], "\n        "))
        
        result[i] = cos_phase[i] * sum_cos + sin_phase[i] * sum_sin
    end
    
    return result
end

end # module
"""

# 保存到文件
open("optimized_polynomials.jl", "w") do io
    write(io, full_code)
end

println("模块已保存到: optimized_polynomials.jl")


模块已保存到: optimized_polynomials.jl


In [22]:

# 加载模块
include("optimized_polynomials.jl")
using .OptimizedPolynomials

# ========== 性能测试函数 ==========
function runloop_optimized!(result, exp_cos, exp_sin, cos_phase, sin_phase,
                            x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    @inbounds for j in 1:n_iterations
        result = OptimizedPolynomials.compute_optimized!(result, exp_cos, exp_sin,
                                               cos_phase, sin_phase,
                                               x, y, z, kx, ky, kz, b, h, mu)
        # open("analysis.txt", "w") do io
        #    println(io, "=== LLVM IR ===")
        #    redirect_stdout(io) do
        #        @code_llvm debuginfo=:none OptimizedPolynomials.compute_optimized!(
        #            result, exp_cos, exp_sin, cos_phase, sin_phase,
        #            x, y, z, kx, ky, kz, b, h, mu
        #        )
        #    end
        # end
        # println("已保存到 analysis.txt")
        # println("文件大小: ", filesize("analysis.txt"), " 字节")                                    
    end
    return result
end


# ========== 测试不同格点数量 ==========
grid_sizes = collect(8000:1000:8000)
times = Float64[]
n_iterations = 3000

println("\n开始性能测试...")
println("=" ^ 60)

for n_grid in grid_sizes
    println("\n测试格点数: $n_grid")
    
    # 截取数据
    x = x_full[1:n_grid]
    y = y_full[1:n_grid]
    z = z_full[1:n_grid]
    
    # 预计算
    println("  预计算 exp 值...")
    exp_values_cos = precompute_exp_terms(x, y, z, func_terms.terms1)
    exp_values_sin = precompute_exp_terms(x, y, z, func_terms.terms2)
    
    # 预计算相位
    phase = @. kx * x + ky * y + kz * z + b
    cos_phase = cos.(phase)
    sin_phase = sin.(phase)
    
    result = zeros(Float64, n_grid)
    
    # 预热
    println("  预热中...")
    runloop_optimized!(result, exp_values_cos, exp_values_sin,
                      cos_phase, sin_phase,
                      x, y, z, kx, ky, kz, b, h, mu, 1)
    
    # 正式计时
    println("  正式计时...")
    elapsed_time = @elapsed runloop_optimized!(result, exp_values_cos, exp_values_sin,
                                              cos_phase, sin_phase,
                                              x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    
    avg_time = elapsed_time / n_iterations * 1000  # 转换为毫秒
    push!(times, elapsed_time)
    
    println("  平均单次耗时: $(round(avg_time, digits=3)) ms")
    println("  总耗时: $(round(elapsed_time, digits=3)) s")
end

println("\n" * "=" ^ 60)
println("测试完成！")

# ========== 保存结果到CSV ==========
using DelimitedFiles

data_matrix = hcat(grid_sizes, times)
writedlm("benchmark_results.csv", 
         vcat(["grid_size" "time_ms"], data_matrix), 
         ',')

println("\n结果已保存到: benchmark_results.csv")
println("\n格点数  |  平均耗时(ms)")
println("-" ^ 30)
for (n, t) in zip(grid_sizes, times)
    println("$n  |  $(round(t, digits=3))")
end


开始性能测试...

测试格点数: 8000
  预计算 exp 值...
  预热中...
  正式计时...
  平均单次耗时: 0.074 ms
  总耗时: 0.221 s

测试完成！

结果已保存到: benchmark_results.csv

格点数  |  平均耗时(ms)
------------------------------
8000  |  0.221


In [21]:
kx, ky

(0.0, 0.0)

In [20]:
result[1]

-0.3017228023624591

In [ ]:
        OptimizedPolynomials.compute_optimized!(result, exp_cos, exp_sin,
                                               cos_phase, sin_phase,
                                               x, y, z, kx, ky, kz, b, h, mu)

In [32]:
# 方法1：检查CPU特性
using Hwloc
println(Hwloc.topology_info())

# 方法2：使用SIMD.jl
using Pkg
Pkg.add("SIMD")
using SIMD

# 检查向量宽度
println("最大向量宽度: ", SIMD.Vec{8,Float64})  # 如果支持会显示信息

# 方法3：直接查询
run(`wmic cpu get caption`)  # Windows
# 或
println(Sys.CPU_NAME)

Machine: 1 (14.79 GB)
 Package: 1 (14.79 GB)
  NUMANode: 1 (14.79 GB)
   L3Cache: 1 (12.0 MB)
    L2Cache: 5 (2.5 MB)
     L1Cache: 8 (48.0 kB)
      Core: 8
       PU: 8
nothing


   Resolving package versions...
   Installed SIMD ─ v3.7.2
    Updating `C:\Users\32194\.julia\environments\v1.12\Project.toml`
  [fdea26ae] + SIMD v3.7.2
    Updating `C:\Users\32194\.julia\environments\v1.12\Manifest.toml`
  [fdea26ae] + SIMD v3.7.2
Precompiling packages...
  13548.4 ms  ✓ SIMD
  1 dependency successfully precompiled in 15 seconds. 99 already precompiled.


最大向量宽度: Vec{8, Float64}


Base.IOError: IOError: could not spawn `wmic cpu get caption`: no such file or directory (ENOENT)

In [9]:
using Pkg
Pkg.add("LoopVectorization")

    Updating registry at `C:\Users\32194\.julia\registries\General.toml`
   Resolving package versions...
   Installed SIMDTypes ──────────────────────── v0.1.0
   Installed BitTwiddlingConvenienceFunctions ─ v0.1.6
   Installed CpuId ──────────────────────────── v0.3.1
   Installed Adapt ──────────────────────────── v4.4.0
   Installed SciMLPublic ────────────────────── v1.0.0
   Installed LayoutPointers ─────────────────── v0.1.17
   Installed OffsetArrays ───────────────────── v1.17.0
   Installed VectorizationBase ──────────────── v0.21.72
   Installed CPUSummary ─────────────────────── v0.2.7
   Installed IfElse ─────────────────────────── v0.1.1
   Installed ManualMemory ───────────────────── v0.1.8
   Installed ThreadingUtilities ─────────────── v0.5.5
   Installed PolyesterWeave ─────────────────── v0.2.2
   Installed HostCPUFeatures ────────────────── v0.1.17
   Installed ArrayInterface ─────────────────── v7.22.0
   Installed LoopVectorization ──────────────── v0.12.173
   In

In [8]:
println("\n" * "=" ^ 60)


In [14]:
# 加载模块
include("optimized_polynomials.jl")
using .OptimizedPolynomials
x = x_full
y = y_full
z = z_full

# ========== 性能测试 ==========
function runloop_optimized!(result, exp_cos, exp_sin, cos_phase, sin_phase,
                            x, y, z, kx, ky, kz, b, h, mu, n_iterations)
    for j in 1:n_iterations
        OptimizedPolynomials.compute_optimized!(result, exp_cos, exp_sin,
                                               cos_phase, sin_phase,
                                               x, y, z, kx, ky, kz, b, h, mu)
    end
    return result
end

result = zeros(Float64, length(x))

println("\n开始性能测试...")
@time runloop_optimized!(result, exp_values_cos, exp_values_sin,
                        cos_phase, sin_phase,
                        x, y, z, kx, ky, kz, b, h, mu, 300)

println("\n预热后再测试...")
@time runloop_optimized!(result, exp_values_cos, exp_values_sin,
                        cos_phase, sin_phase,
                        x, y, z, kx, ky, kz, b, h, mu, 300)

println("\n完成！结果的前10个值：")
println(result[1:10])


开始性能测试...


UndefVarError: UndefVarError: `exp_values_cos` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [76]:
@time runloop_optimized!(result, exp_values_cos, exp_values_sin,
                        cos_phase, sin_phase,
                        x, y, z, kx, ky, kz, b, h, mu, 3000)

  6.167506 seconds


8000-element Vector{Float64}:
 0.29899264806920317
 0.30086253720864103
 0.3033576523225068
 0.3064846267909936
 0.31014729708595795
 0.31412629154836563
 0.3180839799339402
 0.3216023537758981
 0.3242518871102745
 0.32567739660566297
 ⋮
 0.3242518871102745
 0.3216023537758981
 0.3180839799339402
 0.31412629154836563
 0.31014729708595795
 0.3064846267909936
 0.3033576523225068
 0.30086253720864103
 0.29899264806920317

In [42]:
@time runloop_optimized!(result, exp_values_cos, exp_values_sin,
                        cos_phase, sin_phase,
                        x, y, z, kx, ky, kz, b, h, mu, 3000)

  6.481469 seconds


8000-element Vector{Float64}:
 9.632069174648457
 9.405226903075851
 9.205947741538878
 9.033396330066475
 8.886843590461272
 8.765665981588654
 8.669344861053132
 8.597465953261954
 8.54971892387602
 8.525897060648056
 ⋮
 8.54971892387602
 8.597465953261954
 8.669344861053132
 8.765665981588654
 8.886843590461272
 9.033396330066475
 9.205947741538878
 9.405226903075851
 9.632069174648457

In [22]:
for expr in poly_exprs_cos
    parsed = Meta.parse(expr)
    devec = devectorize(parsed)
    print(devec)
    func = eval(Meta.parse("(x, y, z, kx, ky, kz, b, h, mu) -> " * string(devec)))

    push!(POLY_FUNCS_COS, func)
end

((((((((((((((((((((((((((((((((((((((((((((((((((((((((-0.0012 * kx ^ 5) * x - ((0.0012 * kx ^ 4) * ky) * y) - ((0.0012 * kx ^ 4) * kz) * z) - ((0.0024 * kx ^ 3) * ky ^ 2) * x) - ((0.0024 * kx ^ 3) * kz ^ 2) * x) + (9.6e-5 * kx ^ 3) * x ^ 3 + ((6.4e-5 * kx ^ 3) * x) * y ^ 2 + ((6.4e-5 * kx ^ 3) * x) * z ^ 2 + (0.04592 * kx ^ 3) * x) - ((0.0024 * kx ^ 2) * ky ^ 3) * y) - (((0.0024 * kx ^ 2) * ky ^ 2) * kz) * z) - (((0.0024 * kx ^ 2) * ky) * kz ^ 2) * y) + (((0.00016 * kx ^ 2) * ky) * x ^ 2) * y + ((6.4e-5 * kx ^ 2) * ky) * y ^ 3 + (((6.4e-5 * kx ^ 2) * ky) * y) * z ^ 2 + ((0.04592 * kx ^ 2) * ky) * y) - ((0.0024 * kx ^ 2) * kz ^ 3) * z) + (((0.00016 * kx ^ 2) * kz) * x ^ 2) * z + (((6.4e-5 * kx ^ 2) * kz) * y ^ 2) * z + ((6.4e-5 * kx ^ 2) * kz) * z ^ 3 + ((0.04592 * kx ^ 2) * kz) * z) - ((0.0012kx) * ky ^ 4) * x) - (((0.0024kx) * ky ^ 2) * kz ^ 2) * x) + ((6.4e-5kx) * ky ^ 2) * x ^ 3 + (((0.00016kx) * ky ^ 2) * x) * y ^ 2 + (((6.4e-5kx) * ky ^ 2) * x) * z ^ 2 + ((0.04592kx) * ky ^ 2) *

In [20]:
poly_exprs_cos

3-element Vector{String}:
 "-0.0012 * kx .^ 5 .* x - 0.001" ⋯ 2840 bytes ⋯ " .* z .^ 3 - 0.36884 * kz .* z"
 "0.024 * kx .^ 3 .* x + 0.024 * " ⋯ 491 bytes ⋯ "z .* z .^ 3 - 0.4528 * kz .* z"
 "-0.12 * kx .* x - 0.12 * ky .* y - 0.12 * kz .* z"

In [16]:
POLY_FUNCS_COS[1]

#364 (generic function with 1 method)

In [24]:
@time result_opt = runloop_optimized!(exp_values_cos, exp_values_sin,
                                      cos_phase, sin_phase,
                                      x, y, z, kx, ky, kz, b, h, mu, 300)

 10.531855 seconds (446.40 M allocations: 6.652 GiB, 4.63% gc time)


8000-element Vector{Float64}:
   0.29899264806920317
 NaN
   0.3033576523225068
 NaN
   0.31014729708595795
 NaN
   0.3180839799339402
 NaN
 NaN
 NaN
   ⋮
 NaN
   0.3216023537758981
 NaN
   0.31412629154836563
 NaN
   0.3064846267909936
 NaN
   0.30086253720864103
 NaN

In [13]:
cos_phase

8000-element Vector{Float64}:
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 ⋮
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606
 0.955336489125606

In [8]:
POLY_FUNCS_COS

3-element Vector{Function}:
 #190 (generic function with 1 method)
 #193 (generic function with 1 method)
 #196 (generic function with 1 method)

In [7]:
cos_factor_code 

"cos(b + kx .* x + ky .* y + kz .* z)"